# 02 — Temporally Honest Snapshot Construction

## Multi-Resolution Semantic Abstraction over an Evolving Knowledge Hypergraph


## Purpose

This notebook converts the validated Temporal Knowledge Hypergraph (TKH)
into a sequence of temporally honest snapshots.

The goal is to reconstruct the knowledge state available at different years
without introducing future information into earlier snapshots.


---

## Research question addressed

This notebook answers:

> How can an evolving knowledge hypergraph be reconstructed at different
> points in time while preventing temporal leakage?



## Relation to the project pipeline

Previous stage:

```text
01_load_and_validate_data.ipynb
```

verified that:

- nodes contain temporal metadata;
- hyperedges preserve multi-node relationships;
- TKH structure is valid.


This notebook uses those verified properties to construct:

```text
H(2020)
H(2022)
H(2024)
H(2026)
```


where each snapshot represents only the knowledge available at that time.


---

## Temporal assumption

A node is available in snapshot $(t)$ only when:

$[
first\_seen\_year(v) \leq t
]$


A hyperedge is available only when every participating node is available.


This avoids creating incomplete historical relationships.


---

## Important design choice

Hyperedges are never truncated.

Example:

Original hyperedge:

```text
A ─ B ─ C ─ D
```


If only A and B exist in 2020:

Incorrect:

```text
A ─ B
```


Correct:

```text
hyperedge unavailable
```


because truncation changes the meaning of the original relationship.


---

## Outputs

This notebook produces:

```text
node_snapshots
edge_snapshots
snapshot_summary
snapshot_metadata
```

These represent the temporally consistent TKH states.

## Clone repository

In [13]:
from pathlib import Path


REPO_DIR = Path(
    "/content/tkh-hierarchy-project"
)


if not REPO_DIR.exists():

    !git clone https://github.com/mohamadghoroobi/tkh-hierarchy-project.git


%cd /content/tkh-hierarchy-project

/content/tkh-hierarchy-project


## Imports

In [14]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

## Define paths

In [15]:
PROJECT_DIR = Path(
    "/content/tkh-hierarchy-project"
)


DATA_DIR = (
    PROJECT_DIR
    /
    "data"
)


TKH_PATH = (
    DATA_DIR
    /
    "tkh_collection10.json"
)


print(TKH_PATH)

/content/tkh-hierarchy-project/data/tkh_collection10.json


## Load validated TKH

## Load TKH

The previous notebook verified the TKH schema.

Here we reuse the validated representation:

\[
H=(V,E)
\]

where:

- \(V\): nodes
- \(E\): hyperedges

In [16]:
with open(
    TKH_PATH,
    "r",
    encoding="utf-8"
) as f:

    tkh = json.load(f)


nodes = tkh["nodes"]

hyperedges = tkh["hyperedges"]


node_lookup = {

    node["id"]:
    node

    for node in nodes

}


print(
    "Nodes:",
    len(nodes)
)

print(
    "Hyperedges:",
    len(hyperedges)
)

Nodes: 5798
Hyperedges: 1429


## Snapshot years

## Define temporal snapshots

The analysis considers four historical states:

\[
T=\{2020,2022,2024,2026\}
\]

Each snapshot represents the TKH state available at that year.

In [17]:
SNAPSHOT_YEARS = [
    2020,
    2022,
    2024,
    2026
]


SNAPSHOT_YEARS

[2020, 2022, 2024, 2026]

## Node temporal availability

A node appears in snapshot \(t\) if:

$[
first\_seen\_year(v)\leq t
]$


This uses knowledge availability rather than extraction date.

In [18]:
def node_available_at_time(
    node,
    year
):

    return (

        node.get("first_seen_year") is not None

        and

        node["first_seen_year"] <= year

    )

## Build node snapshots

In [19]:
def build_node_snapshot(
    nodes,
    year
):

    return [

        node

        for node in nodes

        if node_available_at_time(
            node,
            year
        )

    ]

## Generate snapshots

In [20]:
node_snapshots = {}


for year in SNAPSHOT_YEARS:

    node_snapshots[year] = (
        build_node_snapshot(
            nodes,
            year
        )
    )


    print(
        year,
        len(node_snapshots[year])
    )

2020 1505
2022 2164
2024 4164
2026 5798


## Node ID snapshots

In [21]:
node_id_snapshots = {}


for year, snapshot_nodes in node_snapshots.items():

    node_id_snapshots[year] = {

        node["id"]

        for node in snapshot_nodes

    }


for year in SNAPSHOT_YEARS:

    print(
        year,
        len(node_id_snapshots[year])
    )

2020 1505
2022 2164
2024 4164
2026 5798


## Hyperedge rule markdown

## Hyperedge temporal availability

A hyperedge remains valid only when all endpoints exist.

For hyperedge:

$[
e=(v_1,v_2,...,v_k)
] $

the availability condition is:

$ [
\forall v_i \in e:
v_i \in V_t
]$


No endpoint removal is performed.

## Hyperedge function

In [22]:
def edge_available_at_time(
    edge,
    available_node_ids
):

    return all(

        member in available_node_ids

        for member in edge["members"]

    )

## Build edge snapshots

In [23]:
def build_edge_snapshot(
    hyperedges,
    available_node_ids
):

    return [

        edge

        for edge in hyperedges

        if edge_available_at_time(
            edge,
            available_node_ids
        )

    ]

## Generate edge snapshots

In [24]:
edge_snapshots = {}


for year in SNAPSHOT_YEARS:

    edge_snapshots[year] = (

        build_edge_snapshot(

            hyperedges,

            node_id_snapshots[year]

        )

    )


    print(
        year,
        len(edge_snapshots[year])
    )

2020 374
2022 529
2024 983
2026 1429


## Snapshot summary

In [25]:
snapshot_summary = []


for year in SNAPSHOT_YEARS:

    arities = [

        len(edge["members"])

        for edge in edge_snapshots[year]

    ]


    snapshot_summary.append({

        "year":
            year,

        "nodes":
            len(node_snapshots[year]),

        "hyperedges":
            len(edge_snapshots[year]),

        "mean_hyperedge_size":
            np.mean(arities)

            if arities

            else 0

    })


snapshot_summary_df = pd.DataFrame(
    snapshot_summary
)


snapshot_summary_df

,year,nodes,hyperedges,mean_hyperedge_size
0,2020,1505,374,5.572193
1,2022,2164,529,5.810964
2,2024,4164,983,6.189217
3,2026,5798,1429,6.248425


## Temporal consistency checks

In [26]:
for i in range(
    len(SNAPSHOT_YEARS)-1
):

    y1 = SNAPSHOT_YEARS[i]

    y2 = SNAPSHOT_YEARS[i+1]


    assert (

        len(node_snapshots[y2])

        >=

        len(node_snapshots[y1])

    )


print(
    "Temporal node consistency verified."
)

Temporal node consistency verified.


## Save metadata

In [27]:
snapshot_metadata = {


    str(year):

    {

        "nodes":
            len(node_snapshots[year]),

        "hyperedges":
            len(edge_snapshots[year]),

        "definition":
            "first_seen_year <= snapshot_year"

    }

    for year in SNAPSHOT_YEARS

}


with open(
    "snapshot_metadata.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        snapshot_metadata,
        f,
        indent=2
    )


print(
    "Snapshot metadata saved."
)

Snapshot metadata saved.


# Section 2 has established

*   TKH temporal snapshots can be reconstructed using knowledge availability rather than extraction metadata.
*   Snapshot membership is defined using:


   $ [
    first\_seen\_year(v) \leq t
  ] $

    ensuring that nodes only appear after they become available.

*   Four temporally consistent TKH snapshots were constructed:

    *   \(H(2020)\)
    *   \(H(2022)\)
    *   \(H(2024)\)
    *   \(H(2026)\)

*   Future node leakage is prevented by excluding nodes whose first appearance occurs after the snapshot year.
*   Hyperedges are preserved as original higher-order relationships and are never converted into pairwise edges.
*   Hyperedges are included in a snapshot only when all endpoint nodes are available at that time.
*   No endpoint truncation is performed; partially available hyperedges are excluded rather than modified.
*   Snapshot construction therefore preserves the semantic meaning of the original hypergraph structure.
*   Temporal consistency checks confirm that later snapshots contain the knowledge state available at their corresponding years.
*   The resulting snapshots provide a temporally honest representation of the evolving TKH.

## Git Push

In [ ]:
from pathlib import Path

CLEAN = Path("/content/drive/MyDrive/Apply/Germany/ConstructorLabs/02_temporal_snapshots.ipynb")
REPO_FILE = Path(
    "/content/tkh-hierarchy-project/"
    "notebooks/02_temporal_snapshots.ipynb"
)

print("Clean file exists:", CLEAN.exists())
print("Repo file exists :", REPO_FILE.exists())